<a href="https://colab.research.google.com/github/mithunkumarsr/NeurIPS-MAS-2026/blob/main/Lab_2_Arbitration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2: Dynamic Goal Arbitration
**NeurIPS 2026 Education Track: Multi-Agent Orchestration**

**Author:** Mithun Kumar S R (Google)

Without external arbitration, multi-agent systems trap each other in non-deterministic deadlocks. In this lab, we resolve infinite loops mathematically using the arbitration equation:
$$ a^* = \arg\max_{a_i \in A} \mathcal{F}(a_i, \omega(t), \mathcal{R}) $$

Reviewer Note: These labs use the native Google Colab Secrets manager. Before running, please click the 'Key' icon on the left sidebar in Colab, create a new secret named GOOGLE_API_KEY, paste your Gemini API key, and toggle 'Notebook access' to ON.

In [ ]:
!pip install -q google-genai

import json
from google import genai
from google.genai import types
from google.colab import userdata

api_key = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=api_key)

### 1. Generating Proposals via Gemini 3.5 Flash
Instead of agents taking immediate action, they generate "Proposals" scored against their specific heuristics. We enforce strict JSON schemas to ensure the Arbitrator can read the outputs mathematically.

In [ ]:
def generate_proposal(persona: str, task: str):
    """Instructs a specialized sub-agent to generate a candidate action."""
    schema = {
        "type": "OBJECT",
        "properties": {
            "agent": {"type": "STRING"},
            "action_type": {"type": "STRING", "description": "fast_patch or safe_qa"},
            "safety_score": {"type": "INTEGER"},
            "speed_score": {"type": "INTEGER"}
        }
    }

    response = client.models.generate_content(
        model='gemini-3.5-flash',
        contents=f"Task: {task}",
        config=types.GenerateContentConfig(
            system_instruction=persona,
            temperature=0.2,
            response_mime_type="application/json",
            response_schema=schema,
        ),
    )
    return json.loads(response.text)

proposal_alpha = generate_proposal("You are Agent Alpha. Return 'fast_patch'. Optimize for speed. Agent name is Alpha.", "Fix DB Timeout")
proposal_beta = generate_proposal("You are Agent Beta. Return 'safe_qa'. Optimize for safety. Agent name is Beta.", "Fix DB Timeout")

proposals_A = [proposal_alpha, proposal_beta]
print(json.dumps(proposals_A, indent=2))

### 2. The Global State ($\omega(t)$) and Arbitration Loop ($a^*$)
The system monitors resources to dynamically shift the evaluation weights, breaking deadlocks when resources are depleted.

In [ ]:
class GlobalState:
    def __init__(self, token_budget):
        self.token_budget = token_budget

    def get_mode(self):
        return "CRISIS" if self.token_budget < 1000 else "NORMAL"

def arbitrate(proposals, omega_t):
    best_score = -float('inf')
    best_action = None
    mode = omega_t.get_mode()

    print(f"--- Arbitration Initiated | State: {mode} ---")
    for p in proposals:
        # Base Heuristic
        score = (p["safety_score"] * 0.5) + (p["speed_score"] * 0.5)

        # Decision Boundary Shift
        if mode == "CRISIS" and p["action_type"] == "fast_patch":
            score += 40.0  # Emergency weight shift
        elif mode == "NORMAL" and p["action_type"] == "safe_qa":
            score += 40.0  # Standard weight shift

        print(f"Evaluating {p['agent']} -> Heuristic Score: {score}")
        if score > best_score:
            best_score = score
            best_action = p

    print(f"\n[VERDICT] Arbitrator mathematically selected: {best_action['agent']}'s proposal.")
    return best_action

# Simulate a depleted budget deadlock
omega_t = GlobalState(token_budget=850)
optimal_action = arbitrate(proposals_A, omega_t)